In [1]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# 개선사항
# 1. 데이터증강 완화 (HorizontalFlip, 약한 ShiftScaleRotate, 약한 RandomBrightnessContrast)
# 2. 해상도 224 → 384
# 3. dev 데이터 가중 방식 WeightedRandomSampler를 사용 ( 데이터 중복에 의한 과적합 감소, dev 가중치 조절이 용이)
# 4. early stopping
# 5. fold 성능 기록이 자동화
# 6. 분류 방식 변경 (NUM_CLASSES = 1), 단일 확률 예측 + BCEWithLogitsLoss()
import os
import gc
import cv2
import math
import pandas as pd
import numpy as np
import random
from tqdm.auto import tqdm
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import WeightedRandomSampler

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm

import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. Configuration
# ==========================================
class CFG:
    # Paths
    BASE_DIR = '/kaggle/input/datasets/ltg2757/taegeon/official_data/open'
    TRAIN_CSV = os.path.join(BASE_DIR, 'train.csv')
    SAMPLE_SUB = os.path.join(BASE_DIR, 'sample_submission.csv')

    TRAIN_DIR = os.path.join(BASE_DIR, 'train')
    TEST_DIR = os.path.join(BASE_DIR, 'test')
    DEV_CSV = os.path.join(BASE_DIR, 'dev.csv')
    DEV_DIR = os.path.join(BASE_DIR, 'dev')

    # Model
    MODEL_NAME = 'convnext_small.fb_in22k_ft_in1k'
    IMG_SIZE = 384
    NUM_CLASSES = 1

    FRAMES_DIR = '/kaggle/input/datasets/ltg2757/data-frames'

    # Training
    EPOCHS = 15
    BATCH_SIZE = 16
    LEARNING_RATE = 2e-4
    WEIGHT_DECAY = 1e-4
    N_FOLDS = 5
    SEED = 42
    NUM_WORKERS = 0

    # Weighted sampler
    USE_WEIGHTED_SAMPLER = True
    DEV_SAMPLE_WEIGHT = 3.0
    TRAIN_SAMPLE_WEIGHT = 1.0

    # Early stopping
    EARLY_STOPPING_PATIENCE = 4
    EARLY_STOPPING_MIN_DELTA = 0 #1e-4

    # Checkpoints
    SAVE_DIR = './models'

os.makedirs(CFG.SAVE_DIR, exist_ok=True)

# ==========================================
# 2. Utils & Metric
# ==========================================
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def binary_log_loss_score(y_true, y_pred, eps=1e-7):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)

    y_pred = np.clip(y_pred, eps, 1.0 - eps)
    loss = -(y_true * np.log(y_pred) + (1.0 - y_true) * np.log(1.0 - y_pred))
    return np.mean(loss)


# ==========================================
# 3. Dataset & Augmentation
# ==========================================
class StructuralDataset(Dataset):
    def __init__(self, df, default_img_dir=None, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.default_img_dir = default_img_dir
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id = row['id']

        img_dir = row['img_dir'] if 'img_dir' in row.index else self.default_img_dir
        is_video = int(row['is_video']) if 'is_video' in row.index else 0

        # =========================
        # 🔥 video frame 처리
        # =========================
        if is_video == 1:
            frame_name = random.choice(['0.png', '1.png', '2.png'])
            frame_path = os.path.join(img_dir, img_id, frame_name)

            frame = cv2.imread(frame_path)
            if frame is None:
                raise FileNotFoundError(f"{frame_path}")

            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            front_img = frame
            top_img = frame

        else:
            front_path = os.path.join(img_dir, img_id, 'front.png')
            top_path = os.path.join(img_dir, img_id, 'top.png')

            front_img = cv2.imread(front_path)
            top_img = cv2.imread(top_path)

            if front_img is None:
                raise FileNotFoundError(front_path)
            if top_img is None:
                raise FileNotFoundError(top_path)

            front_img = cv2.cvtColor(front_img, cv2.COLOR_BGR2RGB)
            top_img = cv2.cvtColor(top_img, cv2.COLOR_BGR2RGB)

        if self.transform:
            augmented = self.transform(image=front_img, top_image=top_img)
            front_img = augmented['image']
            top_img = augmented['top_image']

        if self.is_test:
            return front_img, top_img

        label_str = row['label']
        target = 1.0 if label_str == 'stable' else 0.0

        return front_img, top_img, np.float32(target)


def get_train_transforms():
    return A.Compose([
        A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE),
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(
            shift_limit=0.03,
            scale_limit=0.05,
            rotate_limit=8,
            border_mode=cv2.BORDER_CONSTANT,
            p=0.3
        ),
        A.RandomBrightnessContrast(
            brightness_limit=0.1,
            contrast_limit=0.1,
            p=0.3
        ),
        A.Normalize(mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225]),
        ToTensorV2()
    ], additional_targets={'top_image': 'image'})


def get_valid_transforms():
    return A.Compose([
        A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE),
        A.Normalize(mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225]),
        ToTensorV2()
    ], additional_targets={'top_image': 'image'})


def get_tta_transforms():
    return A.Compose([
        A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE),
        A.HorizontalFlip(p=1.0),
        A.Normalize(mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225]),
        ToTensorV2()
    ], additional_targets={'top_image': 'image'})


# ==========================================
# 4. Model
# ==========================================
class GeM(nn.Module):
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):
        return nn.functional.avg_pool2d(
            x.clamp(min=self.eps).pow(self.p),
            kernel_size=(x.size(-2), x.size(-1))
        ).pow(1.0 / self.p)


class DualBranchModel(nn.Module):
    def __init__(self, model_name=CFG.MODEL_NAME, pretrained=True):
        super().__init__()

        self.backbone_front = timm.create_model(
            model_name,
            pretrained=pretrained,
            num_classes=0,
            global_pool=''
        )
        self.backbone_top = timm.create_model(
            model_name,
            pretrained=pretrained,
            num_classes=0,
            global_pool=''
        )

        in_features = self.backbone_front.num_features
        self.pooling = GeM()

        self.head = nn.Sequential(
            nn.Linear(in_features * 2, 1024),
            nn.BatchNorm1d(1024),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(1024, 1)
        )

    def forward(self, front_img, top_img):
        front_feat = self.pooling(self.backbone_front(front_img)).flatten(1)
        top_feat = self.pooling(self.backbone_top(top_img)).flatten(1)

        x = torch.cat([front_feat, top_feat], dim=1)
        x = self.head(x).squeeze(1)  # (B,)
        return x


# ==========================================
# 5. Train / Valid Functions
# ==========================================
def train_one_epoch(model, loader, optimizer, criterion, device, scaler):
    model.train()
    running_loss = 0.0

    pbar = tqdm(loader, total=len(loader), desc='Train', leave=False)
    for front_img, top_img, targets in pbar:
        front_img = front_img.to(device, non_blocking=True)
        top_img = top_img.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True).float()

        optimizer.zero_grad(set_to_none=True)

        with autocast():
            outputs = model(front_img, top_img)
            loss = criterion(outputs, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * front_img.size(0)
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    epoch_loss = running_loss / len(loader.dataset)
    return epoch_loss


def valid_one_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0

    all_targets = []
    all_probs = []

    pbar = tqdm(loader, total=len(loader), desc='Valid', leave=False)
    with torch.no_grad():
        for front_img, top_img, targets in pbar:
            front_img = front_img.to(device, non_blocking=True)
            top_img = top_img.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True).float()

            with autocast():
                outputs = model(front_img, top_img)
                # loss = criterion(outputs, targets)

            probs = torch.sigmoid(outputs.float())

            # running_loss += loss.item() * front_img.size(0)
            all_targets.extend(targets.detach().cpu().numpy().astype(np.float64))
            all_probs.extend(probs.detach().cpu().numpy().astype(np.float64))

    # val_loss = running_loss / len(loader.dataset)
    val_loss = None
    val_log_loss = binary_log_loss_score(
        np.array(all_targets, dtype=np.float64),
        np.array(all_probs, dtype=np.float64)
    )

    return val_loss, val_log_loss


def train_loop(train_df, fold_idx, valid_df):
    print(f"\n{'=' * 20} Fold: {fold_idx} {'=' * 20}")

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    train_dataset = StructuralDataset(
        train_df,
        default_img_dir=None,
        transform=get_train_transforms(),
        is_test=False
    )
    valid_dataset = StructuralDataset(
        valid_df,
        default_img_dir=None,
        transform=get_valid_transforms(),
        is_test=False
    )

    if CFG.USE_WEIGHTED_SAMPLER:
        sample_weights = train_df['is_dev'].map({
            0: CFG.TRAIN_SAMPLE_WEIGHT,
            1: CFG.DEV_SAMPLE_WEIGHT
        }).values

        sampler = WeightedRandomSampler(
            weights=torch.DoubleTensor(sample_weights),
            num_samples=len(sample_weights),
            replacement=True
        )

        train_loader = DataLoader(
            train_dataset,
            batch_size=CFG.BATCH_SIZE,
            sampler=sampler,
            num_workers=CFG.NUM_WORKERS,
            drop_last=True,
            pin_memory=torch.cuda.is_available()
        )
    else:
        train_loader = DataLoader(
            train_dataset,
            batch_size=CFG.BATCH_SIZE,
            shuffle=True,
            num_workers=CFG.NUM_WORKERS,
            drop_last=True,
            pin_memory=torch.cuda.is_available()
        )

    valid_loader = DataLoader(
        valid_dataset,
        batch_size=CFG.BATCH_SIZE,
        shuffle=False,
        num_workers=CFG.NUM_WORKERS,
        pin_memory=torch.cuda.is_available()
    )

    model = DualBranchModel(pretrained=True).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=CFG.LEARNING_RATE,
        weight_decay=CFG.WEIGHT_DECAY
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer,
        T_0=CFG.EPOCHS,
        T_mult=1,
        eta_min=1e-6
    )

    scaler = GradScaler()
    best_loss = np.inf
    patience_counter = 0

    for epoch in range(CFG.EPOCHS):
        print(f"\n[Fold {fold_idx}] Epoch {epoch + 1}/{CFG.EPOCHS}")

        train_loss = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
            device=device,
            scaler=scaler
        )

        val_loss, val_log_loss = valid_one_epoch(
            model=model,
            loader=valid_loader,
            criterion=criterion,
            device=device
        )

        scheduler.step()

        current_lr = optimizer.param_groups[0]['lr']
        print(
            f"Fold {fold_idx} | "
            f"Train Loss: {train_loss:.6f} | "
            # f"Val Loss: {val_loss:.6f} | "
            f"Val LogLoss: {val_log_loss:.6f} | "
            f"LR: {current_lr:.8f}"
        )

        if val_log_loss < best_loss - CFG.EARLY_STOPPING_MIN_DELTA:
            best_loss = val_log_loss
            patience_counter = 0

            save_path = os.path.join(CFG.SAVE_DIR, f'best_fold{fold_idx}.pth')
            torch.save(model.state_dict(), save_path)
            print(f"  -> Best model saved to {save_path}")
        else:
            patience_counter += 1
            print(f"  -> No improvement. patience = {patience_counter}/{CFG.EARLY_STOPPING_PATIENCE}")

            if patience_counter >= CFG.EARLY_STOPPING_PATIENCE:
                print("  -> Early stopping triggered.")
                break

    del model, train_loader, valid_loader, train_dataset, valid_dataset
    gc.collect()
    torch.cuda.empty_cache()

    return best_loss


# ==========================================
# 6. Inference
# ==========================================
def predict_with_loader(model, loader, device):
    preds = []

    model.eval()
    with torch.no_grad():
        for front_img, top_img in tqdm(loader, total=len(loader), leave=False):
            front_img = front_img.to(device, non_blocking=True)
            top_img = top_img.to(device, non_blocking=True)

            with autocast():
                outputs = model(front_img, top_img)
                # loss = criterion(outputs, targets)

            probs = torch.sigmoid(outputs.float())
            preds.extend(probs.tolist())

    return np.array(preds, dtype=np.float32)


# ==========================================
# 7. Main Pipeline
# ==========================================
def main_pipeline():
    seed_everything(CFG.SEED)

    # =========================
    # 1. Load train/dev
    # =========================
    train_df_raw = pd.read_csv(CFG.TRAIN_CSV)
    train_df_raw['img_dir'] = CFG.TRAIN_DIR
    train_df_raw['is_dev'] = 0
    train_df_raw['is_video'] = 0

    dev_df_raw = pd.read_csv(CFG.DEV_CSV)
    dev_df_raw['img_dir'] = CFG.DEV_DIR
    dev_df_raw['is_dev'] = 1
    dev_df_raw['is_video'] = 0

    # =========================
    # 2. 🔥 video frame 추가
    # =========================
    VIDEO_FRAME_DIR = "/kaggle/input/datasets/ltg2757/data-frames"

    video_ids = os.listdir(VIDEO_FRAME_DIR)

    video_df = pd.DataFrame({'id': video_ids})

    # label 붙이기 (train 기준)
    video_df = video_df.merge(
        train_df_raw[['id', 'label']],
        on='id',
        how='left'
    )

    video_df = video_df.dropna(subset=['label']).reset_index(drop=True)

    video_df['img_dir'] = VIDEO_FRAME_DIR
    video_df['is_dev'] = 0
    video_df['is_video'] = 1

    print(f"Video samples: {len(video_df)}")

    # =========================
    # 3. concat
    # =========================
    full_df = pd.concat(
        [train_df_raw, dev_df_raw, video_df],
        ignore_index=True
    )

    full_df['fold'] = -1

    print(f"Total samples: {len(full_df)}")

    # =========================
    # 4. Stratified KFold
    # =========================
    skf = StratifiedKFold(
        n_splits=CFG.N_FOLDS,
        shuffle=True,
        random_state=CFG.SEED
    )

    for fold, (_, val_idx) in enumerate(skf.split(full_df, full_df['label'])):
        full_df.loc[val_idx, 'fold'] = fold

    # =========================
    # 5. Train
    # =========================
    fold_log_losses = []

    for fold in range(CFG.N_FOLDS):
        train_data = full_df[full_df['fold'] != fold].reset_index(drop=True)
        valid_data = full_df[full_df['fold'] == fold].reset_index(drop=True)

        best_loss = train_loop(train_data, fold, valid_data)
        fold_log_losses.append(best_loss)

    print("fold_log_losses:", fold_log_losses)

    # for fold in range(CFG.N_FOLDS):
    #     train_data = full_df[full_df['fold'] != fold].reset_index(drop=True)
    #     valid_data = full_df[full_df['fold'] == fold].reset_index(drop=True)

    #     print(f"\nFold {fold}: train={len(train_data)}, valid={len(valid_data)}")
    #     print("Train label dist:")
    #     print(train_data['label'].value_counts())
    #     print("Valid label dist:")
    #     print(valid_data['label'].value_counts())

    #     best_loss = train_loop(train_data, fold, valid_data)
    #     fold_log_losses.append(best_loss)

    print("\nfold_log_losses:", fold_log_losses)
    print("\nTraining complete. Start inference...")

    # 3) Test loaders
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    sub_df = pd.read_csv(CFG.SAMPLE_SUB)

    test_dataset_base = StructuralDataset(
        sub_df,
        default_img_dir=CFG.TEST_DIR,
        transform=get_valid_transforms(),
        is_test=True
    )
    test_loader_base = DataLoader(
        test_dataset_base,
        batch_size=CFG.BATCH_SIZE * 2,
        shuffle=False,
        num_workers=CFG.NUM_WORKERS,
        pin_memory=torch.cuda.is_available()
    )

    test_dataset_tta = StructuralDataset(
        sub_df,
        default_img_dir=CFG.TEST_DIR,
        transform=get_tta_transforms(),
        is_test=True
    )
    test_loader_tta = DataLoader(
        test_dataset_tta,
        batch_size=CFG.BATCH_SIZE * 2,
        shuffle=False,
        num_workers=CFG.NUM_WORKERS,
        pin_memory=torch.cuda.is_available()
    )

    loaders = [test_loader_base, test_loader_tta]

          
   
   

       # --------------------------
    # Fold ensemble weights 계산
    # --------------------------
    # 1) inverse of log losses
    inv_losses = np.array([1.0 / (loss + 1e-6) for loss in fold_log_losses])
    
    # 2) overflow 방지 (소프트맥스 안정화)
    inv_losses = inv_losses - np.max(inv_losses)
    
    # 3) softmax normalization
    exp_weights = np.exp(inv_losses)
    norm_weights = exp_weights / np.sum(exp_weights)
    
    # ✅ 계산 후 출력
    print("Fold ensemble weights:", norm_weights)
    
   

    ensemble_stable_preds = np.zeros(len(sub_df), dtype=np.float32)

    for fold in range(CFG.N_FOLDS):
        model = DualBranchModel(pretrained=False).to(device)
        ckpt_path = os.path.join(CFG.SAVE_DIR, f'best_fold{fold}.pth')
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
        model.eval()

        fold_preds = np.zeros(len(sub_df), dtype=np.float32)

        for loader_idx, loader in enumerate(loaders):
            tta_preds = predict_with_loader(model, loader, device)
            fold_preds += tta_preds / len(loaders)

        ensemble_stable_preds += fold_preds * norm_weights[fold]

        del model
        gc.collect()
        torch.cuda.empty_cache()

    # 5) Submission formatting
    ensemble_stable_preds = np.clip(ensemble_stable_preds, 1e-15, 1 - 1e-15)
    ensemble_unstable_preds = 1.0 - ensemble_stable_preds

    sub_df['unstable_prob'] = ensemble_unstable_preds
    sub_df['stable_prob'] = ensemble_stable_preds

    sub_df.to_csv('submission.csv', index=False)
    print("Ensemble inference finished! submission.csv saved.")


if __name__ == '__main__':
    main_pipeline()

Video samples: 1000
Total samples: 2100

==================== Fold: 0 ====================


model.safetensors:   0%|          | 0.00/201M [00:00<?, ?B/s]


[Fold 0] Epoch 1/15


Train:   0%|          | 0/105 [00:00<?, ?it/s]

Valid:   0%|          | 0/27 [00:00<?, ?it/s]

...

  0%|          | 0/32 [00:00<?, ?it/s]

  0%|          | 0/32 [00:00<?, ?it/s]

  0%|          | 0/32 [00:00<?, ?it/s]

  0%|          | 0/32 [00:00<?, ?it/s]

Ensemble inference finished! submission.csv saved.
